# U.S. State Crime Rate | Census + NIBRS Data Pipeline

**What this notebook does:**
1. Mounts Google Drive and extracts the NIBRS ZIP
2. Loads all 11.2M NIBRS incident rows and aggregates to state level
3. Pulls 2022 Census ACS socioeconomic data via API
4. Joins both datasets on state FIPS code
5. Saves `state_crime_census_2022.csv` locally and to shared Drive

**Run cells top to bottom. Total runtime ~5 minutes.**

In [ ]:
# Install dependencies
!pip install requests pandas -q

import requests, time, zipfile
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Paths
DRIVE_FOLDER = Path('/content/drive/Shareddrives/MSBA Group 13/Big Data /BUAD5722_Crime_Project')
NIBRS_FILE   = Path('/content/crime_project_data/nibrs_raw/ICPSR_38925/DS0002/38925-0002-Data.tsv')
CENSUS_DIR   = Path('/content/crime_project_data/census_raw')
OUTPUT_DIR   = Path('/content/crime_project_data/clean')

CENSUS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Ready.')

In [ ]:
# Extract NIBRS ZIP from shared Drive
# Skips if already extracted (safe to re-run)
NIBRS_DIR = Path('/content/crime_project_data/nibrs_raw')
NIBRS_DIR.mkdir(parents=True, exist_ok=True)

if NIBRS_FILE.exists():
    print('Already extracted.')
else:
    zip_path = DRIVE_FOLDER / 'ICPSR_38925-V2.zip'
    print(f'Extracting {zip_path.name}... (may take 2-3 min)')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(NIBRS_DIR)
    print('Done.')

In [ ]:
# Load NIBRS administrative file in 500k-row chunks to avoid memory crashes
# V-codes are raw ICPSR column names; renamed to readable labels via COL_NAMES

COL_NAMES = {
    'V1001': 'ori',
    'V1002': 'incident_number',
    'FIPS_STATE': 'fips_state',
    'V1003': 'agency_state',
    'V1004': 'population_group',
    'V1005': 'division',
    'V1006': 'region',
    'V1007': 'agency_indicator',
    'V1008': 'core_city',
    'V1009': 'covered_by_ori',
    'V1010': 'num_months_reported',
    'V1011': 'incident_date',
    'V1012': 'report_date_indicator',
    'V1013': 'incident_hour',
    'V1014': 'cleared_exceptionally',
    'V1015': 'num_offenses',
    'V1016': 'num_victims',
}

KEEP_COLS = ['ori', 'fips_state', 'agency_state', 'num_months_reported',
             'incident_date', 'cleared_exceptionally', 'num_offenses', 'num_victims']

chunks = []
total  = 0

for chunk in pd.read_csv(NIBRS_FILE, sep='\t', chunksize=500_000,
                          low_memory=False, dtype=str):
    chunk = chunk.rename(columns=COL_NAMES)[KEEP_COLS]
    chunk = chunk.dropna(subset=['ori'])
    chunk['incident_date']       = pd.to_datetime(chunk['incident_date'], format='%d-%b-%Y', errors='coerce')
    chunk['year']                = chunk['incident_date'].dt.year
    chunk['num_offenses']        = pd.to_numeric(chunk['num_offenses'], errors='coerce')
    chunk['num_victims']         = pd.to_numeric(chunk['num_victims'],  errors='coerce')
    chunk['num_months_reported'] = pd.to_numeric(chunk['num_months_reported'], errors='coerce')
    chunks.append(chunk)
    total += len(chunk)
    print(f'  {total:,} rows loaded...')

nibrs = pd.concat(chunks, ignore_index=True)
print(f'\nDone. Shape: {nibrs.shape}')

In [ ]:
# Aggregate NIBRS to one row per state
# Note: incident_date parsing failed (all NaT) so no year filter is applied

nibrs['cleared']      = (nibrs['cleared_exceptionally'] != 'N').astype(int)
nibrs['num_offenses'] = pd.to_numeric(nibrs['num_offenses'], errors='coerce').fillna(1)
nibrs['num_victims']  = pd.to_numeric(nibrs['num_victims'],  errors='coerce').fillna(1)

state_agg = nibrs.groupby('fips_state').agg(
    total_incidents   = ('ori',          'count'),
    total_offenses    = ('num_offenses',  'sum'),
    total_victims     = ('num_victims',   'sum'),
    cleared_incidents = ('cleared',       'sum'),
    num_agencies      = ('ori',          'nunique'),
).reset_index()

state_agg['clearance_rate'] = (
    state_agg['cleared_incidents'] / state_agg['total_incidents']
).round(4)

# Zero-pad FIPS to 2 digits so it matches the Census join key
state_agg['fips_state'] = state_agg['fips_state'].astype(str).str.zfill(2)

print(f'States: {len(state_agg)}')
display(state_agg.head())

In [ ]:
# Pull Census ACS 5-year 2022 data via API
# Variables: income, unemployment, poverty, education, population, race, housing

CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

ACS_VARS = {
    'DP03_0062E':  'median_household_income',
    'DP03_0009PE': 'unemployment_rate_pct',
    'DP03_0128PE': 'poverty_rate_pct',
    'DP02_0068PE': 'bachelors_degree_or_higher_pct',
    'DP05_0001E':  'total_population',
    'DP05_0037PE': 'white_alone_pct',
    'DP05_0038PE': 'black_alone_pct',
    'DP05_0071PE': 'hispanic_latino_pct',
    'DP04_0003PE': 'housing_vacancy_rate_pct',
}

var_list = ','.join(['NAME'] + list(ACS_VARS.keys()))
url = (
    f'https://api.census.gov/data/2022/acs/acs5/profile'
    f'?get={var_list}&for=state:*&key={CENSUS_API_KEY}'
)

r = requests.get(url, timeout=60)
r.raise_for_status()
data = r.json()

census = pd.DataFrame(data[1:], columns=data[0])
census = census.rename(columns=ACS_VARS)
census['fips_state'] = census['state'].str.zfill(2)

# Convert to numeric; replace Census missing-value sentinels with NaN
num_cols  = list(ACS_VARS.values())
SENTINELS = [-666666666, -888888888, -999999999, -222222222, -333333333]
for col in num_cols:
    census[col] = pd.to_numeric(census[col], errors='coerce')
census[num_cols] = census[num_cols].replace(SENTINELS, float('nan'))

print(f'Census shape: {census.shape}')
display(census.head())

In [ ]:
# Join NIBRS state aggregation with Census on FIPS code
# Inner join drops territories not in both datasets

combined = state_agg.merge(census, on='fips_state', how='inner')

# Crime rate per 100k is the main modeling target
combined['crime_rate_per_100k'] = (
    combined['total_incidents'] / combined['total_population'] * 100_000
).round(2)

print(f'Final shape: {combined.shape}')
display(
    combined[['NAME', 'total_incidents', 'crime_rate_per_100k',
              'poverty_rate_pct', 'unemployment_rate_pct', 'median_household_income']]
    .sort_values('crime_rate_per_100k', ascending=False)
    .head(10)
)

In [ ]:
# Save output locally and to shared Drive

out_local = OUTPUT_DIR / 'state_crime_census_2022.csv'
combined.to_csv(out_local, index=False)
print(f'Saved locally: {out_local}')

try:
    combined.to_csv(DRIVE_FOLDER / 'state_crime_census_2022.csv', index=False)
    print('Saved to shared Drive.')
except Exception as e:
    print(f'Drive save failed: {e}')

print(f'\nRows: {len(combined)} states')
print(f'Columns: {list(combined.columns)}')